# sigreg-video-lejepa — Pretraining Notebook (Phase 4 / Phase 5)

Runs UCF101 pretraining with automatic HF Hub checkpoint persistence so
training resumes across Kaggle session boundaries.

Supports both Phase 4 experiments (64×64, 25k steps) and Phase 5 experiments
(128×128, 75k steps). Set `EXPERIMENT_NAME` in Cell 6 to select which run to train.

**Workflow (per session):**
1. Set `EXPERIMENT_NAME` in Cell 6 to the run you want to train.
2. Run all cells top-to-bottom. Cell 5 (benchmark) only needs to run once ever.
3. Cell 6 resumes from the latest HF checkpoint automatically.
4. When `max_steps` is reached, run Cells 7-8 to evaluate and push results.

**Prereqs (Kaggle):**
- Dataset `pevogam/ucf101` added to this notebook (Settings → Data).
- `WANDB_API_KEY` and `HF_TOKEN` set in Kaggle Secrets (Settings → Secrets).

In [ ]:
# Cell 1: Clone repo + install deps
# TPU vs GPU mode — set to 'tpu' when running on Kaggle TPU v5e-8 session.
ACCELERATOR = 'gpu'  # or 'tpu'

import os
if ACCELERATOR == 'tpu':
    os.environ['PJRT_DEVICE'] = 'TPU'
    !pip install -q 'torch_xla[tpu]>=2.8.0'


REPO_DIR = '/kaggle/working/sigreg-video-lejepa'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ankitpani8/sigreg-video-lejepa.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!git log --oneline -3
!pip install -q uv
!uv pip install --system -e '.[dev]'

In [ ]:
# Cell 2: Auth — WandB and HF Hub via Kaggle Secrets
# Set these secrets in Kaggle: Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

import os
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN']      = secrets.get_secret('HF_TOKEN')

import wandb, huggingface_hub
wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
huggingface_hub.login(token=os.environ['HF_TOKEN'])
print('Auth OK')

In [ ]:
# Cell 3: UCF101 path setup
#
# pevogam/ucf101 dataset structure on Kaggle:
#   /kaggle/input/ucf101/UCF101/UCF-101/                     ← video files
#   /kaggle/input/ucf101/UCF101/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist/
#
# (The linprobe notebook used a different path; the above matches the actual dataset layout.)
UCF101_DATA   = '/kaggle/input/ucf101/UCF101/UCF-101'
UCF101_SPLITS = '/kaggle/input/ucf101/UCF101/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist'
HF_REPO_ID    = 'ankitpani/sigreg-video-lejepa-checkpoints'
WANDB_PROJECT = 'sigreg-video-lejepa'

import os
print(f'UCF101 data exists:   {os.path.exists(UCF101_DATA)}')
print(f'UCF101 splits exist:  {os.path.exists(UCF101_SPLITS)}')

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 4: Sanity check — decode one video to confirm decord is working
import os, glob, random
video_files = glob.glob(os.path.join(UCF101_DATA, '*', '*.avi'))
print(f'Found {len(video_files)} videos')
sample = random.sample(video_files, 3)
for f in sample:
    print(' ', f)

# Decode the first sample with decord
import decord
vr = decord.VideoReader(sample[0], ctx=decord.cpu(0))
print(f'Decoded {len(vr)} frames, shape: {vr[0].shape}')

In [ ]:
# Cell 5: Throughput benchmark (run once, then skip on subsequent sessions)
#
# Measures avg sec/step over steps 101-1000 with 16-mixed precision.
# SIGReg mode is used; EMA adds ~5% overhead (noted in phase4_benchmark.yaml).
#
# Decision guide:
#   sec/step < 1.0  →  can afford 50k steps per seed
#   sec/step 1-2    →  stay at 25k steps (as planned)
#   sec/step > 2.0  →  cut to 15-20k steps

!cd {REPO_DIR} && python scripts/pretrain.py \
    +experiment=phase4_benchmark \
    data.dataset.data_root={UCF101_DATA} \
    data.dataset.split_root={UCF101_SPLITS} \
    data.dataset.local_cache=null

## GPU vs TPU mode selection

Set `ACCELERATOR = 'gpu'` (in Cell 1) for a **Kaggle GPU session** (T4×2, fp16).
Set `ACCELERATOR = 'tpu'` for a **Kaggle TPU v5e-8 session** (8 chips, bf16).

The `EXPERIMENT_NAME` below is auto-suffixed with `_tpu` when on TPU, selecting the
matching experiment config (`phase5_sigreg_seed0_tpu`, etc.).

**TPU experiment names:** `phase5_sigreg_seed0_tpu`, `phase5_sigreg_seed1_tpu`,
`phase5_ema_seed0_tpu`, `phase5_ema_seed1_tpu`

**Note:** fp16 stays on GPU (T4 has native fp16 tensor cores). bf16 is forced on TPU (v5e-8 does not support fp16 at the hardware level).

In [ ]:
# Cell 6: Main training run
#
# Set EXPERIMENT_NAME to one of:
#   Phase 4 (64×64, 25k steps — complete):
#     phase4_sigreg_seed0, phase4_sigreg_seed1
#     phase4_ema_seed0,    phase4_ema_seed1
#   Phase 5 (128×128, 75k steps — active):
#     phase5_sigreg_seed0, phase5_sigreg_seed1
#     phase5_ema_seed0,    phase5_ema_seed1
#
# Cache coexistence: Phase 5 configs use local_cache=/content/ucf101_local_128
# (vs Phase 4's /content/ucf101_local), so both can coexist on disk if needed.
# This cell passes local_cache=null, disabling caching regardless of config.
#
# Re-run this cell in a new session to resume — it picks up from the latest
# HF checkpoint automatically when resume_from_hf=true.

EXPERIMENT_NAME = f"phase5_sigreg_seed0{'_tpu' if ACCELERATOR == 'tpu' else ''}"  # ← change base name for each run

!cd {REPO_DIR} && python scripts/pretrain.py \
    +experiment={EXPERIMENT_NAME} \
    +run_name={EXPERIMENT_NAME} \
    +hf_repo_id={HF_REPO_ID} \
    +resume_from_hf=true \
    +checkpoint_every_n_steps=1000 \
    +wandb_project={WANDB_PROJECT} \
    data.dataset.data_root={UCF101_DATA} \
    data.dataset.split_root={UCF101_SPLITS} \
    data.dataset.local_cache=null

In [ ]:
# Cell 7: Linear probe evaluation on the just-trained checkpoint
#
# Requires Cell 6 to have completed max_steps. The most recent local checkpoint
# (checkpoints/last.ckpt) is used as the encoder weights.

import subprocess, re, json, os

FEATURES_DIR = f'/kaggle/working/features_{EXPERIMENT_NAME}'
os.makedirs(FEATURES_DIR, exist_ok=True)
CKPT_PATH = f'{REPO_DIR}/checkpoints/last.ckpt'

# Stage 1: extract features
result = subprocess.run(
    ['python', 'scripts/extract_features.py',
     '+experiment=ucf101_linprobe',
     f'checkpoint_path={CKPT_PATH}',
     f'evaluation.features_dir={FEATURES_DIR}',
     f'data.dataset.data_root={UCF101_DATA}',
     f'data.dataset.split_root={UCF101_SPLITS}',
     'data.dataset.local_cache=null',
     'data.num_workers=4'],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('extract_features failed')

# Stage 2: train probe
result2 = subprocess.run(
    ['python', 'scripts/linear_probe.py',
     '+experiment=ucf101_linprobe',
     f'evaluation.features_dir={FEATURES_DIR}',
     'trainer.max_epochs=20',
     'trainer.accelerator=auto'],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(result2.stdout[-3000:])
if result2.returncode != 0:
    print('STDERR:', result2.stderr[-1000:])
    raise RuntimeError('linear_probe failed')

# Parse and save results
m = re.search(r'top-1:\s+([0-9.]+)\s+top-5:\s+([0-9.]+)', result2.stdout)
if m:
    top1, top5 = float(m.group(1)), float(m.group(2))
    linprobe_results = {'experiment': EXPERIMENT_NAME, 'top1': top1, 'top5': top5}
    results_path = f'/kaggle/working/linprobe_{EXPERIMENT_NAME}.json'
    with open(results_path, 'w') as f:
        json.dump(linprobe_results, f, indent=2)
    print(f'\nResults: top-1={top1:.4f}  top-5={top5:.4f}')
else:
    print('WARNING: could not parse linear probe output')

In [ ]:
# Cell 8: Push final checkpoint + linear probe results to HF Hub

from huggingface_hub import HfApi
import os, glob

api = HfApi()
hf_token = os.environ['HF_TOKEN']
results_path = f'/kaggle/working/linprobe_{EXPERIMENT_NAME}.json'

# Push linear probe results JSON
if os.path.exists(results_path):
    api.upload_file(
        path_or_fileobj=results_path,
        path_in_repo=f'results/{EXPERIMENT_NAME}/linprobe.json',
        repo_id=HF_REPO_ID,
        repo_type='model',
        token=hf_token,
    )
    print(f'Pushed results/{EXPERIMENT_NAME}/linprobe.json')
else:
    print(f'Results file not found: {results_path} — run Cell 7 first')

# Push final checkpoint (last.ckpt) explicitly with a labelled name
final_ckpt = f'{REPO_DIR}/checkpoints/last.ckpt'
if os.path.exists(final_ckpt):
    api.upload_file(
        path_or_fileobj=final_ckpt,
        path_in_repo=f'checkpoints/{EXPERIMENT_NAME}/final.ckpt',
        repo_id=HF_REPO_ID,
        repo_type='model',
        token=hf_token,
    )
    print(f'Pushed checkpoints/{EXPERIMENT_NAME}/final.ckpt')

print(f'\nAll artifacts at: https://huggingface.co/{HF_REPO_ID}')